# Stable Diffusion

## From using a pipeline to implementing the diffusion algorithm

In this exercise, you will work with a pretrained Stable Diffusion model, but **you will implement important parts of the inference algorithm yourself**. Instead of calling the pipeline for every experiment, you will directly use the tokenizer, text encoder, VAE, U-Net, and scheduler.

We use pretrained weights because training Stable Diffusion from scratch is beyond the scope of this exercise. Your implementation will focus on the inference loop and the core mechanisms that make text-to-image diffusion work.



### Learning goals
By the end, you should be able to:
- explain the latent diffusion pipeline and tensor shapes;
- implement classifier-free guidance (CFG);
- implement the iterative denoising loop rather than relying on `pipe(prompt)`;
- inspect the effect of scheduler updates, seeds, and inference steps;
- compare a custom loop against the reference Diffusers pipeline;
- reason about runtime and memory trade-offs.

### What you will implement
1. Text conditioning and unconditional conditioning.
2. Latent initialization and the denoising loop.
3. Classifier-free guidance.
4. A reusable generation function.
5. Experiments and quantitative checks.

Cells marked **🎯TODO** are what you need to complete

## 1. Environment and imports

Install dependencies, then check that a GPU is available. If you use Colab, select **Runtime → Change runtime type → GPU**.

In [ ]:
%pip -q install diffusers transformers accelerate safetensors matplotlib

In [ ]:
import time
import torch
import matplotlib.pyplot as plt
from diffusers import StableDiffusionPipeline, DDIMScheduler

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Warning: this exercise will be very slow on CPU.")

## 2. Load components, not just the pipeline

We load a pipeline once to obtain pretrained components. In the core exercise, we will call those components directly and write the sampling loop ourselves.

Stable Diffusion 1.5 typically uses:
- tokenizer + text encoder: prompt → embeddings;
- U-Net: noisy latent + timestep + text embeddings → predicted noise;
- scheduler: predicted noise → updated latent;
- VAE decoder: final latent → image.

The pipeline's safety checker is disabled here to keep the exercise focused on model mechanics


In [ ]:
model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"
dtype = torch.float16 if device == "cuda" else torch.float32

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=dtype,
    safety_checker=None,
    requires_safety_checker=False,
).to(device)

# Use DDIM for a transparent, deterministic-style sampling setup.
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
pipe.set_progress_bar_config(disable=False)

tokenizer = pipe.tokenizer
text_encoder = pipe.text_encoder
unet = pipe.unet
vae = pipe.vae
scheduler = pipe.scheduler

print("U-Net input channels:", unet.config.in_channels)
print("Latent channels:", vae.config.latent_channels)
print("VAE scaling factor:", vae.config.scaling_factor)

### Task 1 — Inspect the model

Run the cell and explain the values printed. Then inspect the shape of the tokenized prompt and the output of the text encoder.



In [ ]:
prompt = "A lighthouse on a rocky coast during a storm, oil painting"
tokens = tokenizer(
    prompt,
    padding="max_length",
    max_length=tokenizer.model_max_length,
    truncation=True,
    return_tensors="pt",
).input_ids.to(device)

with torch.no_grad():
    #🎯TODO: Use the text encoder to obtain text embeddings from the token IDs.
    text_embeddings = ...[0]

print("Token IDs:", tokens.shape)
print("Text embeddings:", text_embeddings.shape)


## 3. Implement text conditioning and classifier-free guidance

Classifier-free guidance (CFG) runs the U-Net twice at each denoising step:
- once with the real prompt embedding, giving `noise_cond`;
- once with an empty prompt embedding, giving `noise_uncond`.

The guided prediction is

$$
\hat{\epsilon} = \epsilon_{\mathrm{uncond}} + s(\epsilon_{\mathrm{cond}} - \epsilon_{\mathrm{uncond}})
$$
where \(s\) is `guidance_scale`. To save a U-Net call, implementations often concatenate the unconditional and conditional batches and run them together.



In [ ]:
def encode_prompt(prompt, tokenizer, text_encoder, device):
    """Return CFG embeddings [unconditional, conditional]."""
    texts = ["", prompt]
    token_ids = tokenizer(
        texts,
        padding="max_length",
        max_length=tokenizer.model_max_length,
        truncation=True,
        return_tensors="pt",
    ).input_ids.to(device)
    embeddings = text_encoder(token_ids)[0]
    return embeddings


## 4. Implement the latent denoising loop

At each timestep:
1. Duplicate the latent batch so unconditional and conditional predictions can be made together.
2. Scale the model input as required by the scheduler.
3. Predict noise with the U-Net.
4. Split unconditional and conditional noise predictions.
5. Apply CFG.
6. Ask the scheduler for the previous, less-noisy latent.

The scheduler must be configured with `set_timesteps` for the chosen number of inference steps. The initial latent is sampled from a Gaussian and scaled by `scheduler.init_noise_sigma`.



In [ ]:
@torch.no_grad()
def generate_latents(
    prompt,
    height=512,
    width=512,
    num_inference_steps=25,
    guidance_scale=7.5,
    seed=0,
):
    """Generate a latent with a hand-written Stable Diffusion sampling loop."""
    assert height % 8 == 0 and width % 8 == 0

    # 🎯TODO: Encode the text prompt into embeddings for classifier-free guidance
    # using the encode_prompt function
    embeddings = ...
    scheduler.set_timesteps(num_inference_steps, device=device)

    latent_h, latent_w = height // 8, width // 8
    generator = torch.Generator(device=device).manual_seed(seed)
    latents = torch.randn(
        (1, unet.config.in_channels, latent_h, latent_w),
        generator=generator, device=device, dtype=embeddings.dtype,
    )
    # 🎯TODO: Scale the initial noise according to the scheduler's expected noise level.
    # Hint: Multiply `latents` by the scheduler's `init_noise_sigma`.
    latents = ...

    for t in scheduler.timesteps:
        latent_model_input = torch.cat([latents, latents], dim=0)
        latent_model_input = scheduler.scale_model_input(latent_model_input, t)
        noise_pred = unet(
            latent_model_input, t, encoder_hidden_states=embeddings
        ).sample
        noise_uncond, noise_cond = noise_pred.chunk(2)
        # 🎯TODO: Implement the guided_noise according to the guidance prediction
        # defined as ϵ^=ϵuncond+s(ϵcond−ϵuncond) in step 3
        guided_noise = ...
        # 🎯TODO: Use the scheduler to update the latents for the next timestep.
        # Hint: Call `scheduler.step()` with the predicted noise (`guided_noise`),
        # the current timestep (`t`), and the current latents (`latents`).
        latents = ... .prev_sample

    return latents


## 5. Decode latents into an image

The VAE decoder maps the final latent back to RGB pixels. Stable Diffusion uses a scaling factor on latents; divide by that factor before decoding.



In [ ]:
@torch.no_grad()
def decode_latents(latents):
    # 🎯TODO: Rescale the latents before passing them to the VAE decoder.
    #Hint: you can use the scaling_factor from vae.config
    latents = ...
    image = vae.decode(latents.to(dtype=vae.dtype)).sample
    image = (image / 2 + 0.5).clamp(0, 1)
    image = image[0].permute(1, 2, 0).float().cpu().numpy()
    from PIL import Image
    image = (image * 255).round().astype("uint8")
    return Image.fromarray(image)


## 6. Bring your implementation together

Once Tasks 1–4 are complete, run this cell. If the implementation is correct, the output should be a plausible image related to the prompt. It will not necessarily be pixel-identical to the reference pipeline because implementations and scheduler settings can differ.

In [ ]:
prompt = "A tiny bookstore in a rainy European street, warm lights, cinematic photography"

start = time.perf_counter()
latents = generate_latents(
    prompt,
    num_inference_steps=25,
    guidance_scale=7.5,
    seed=123,
)
custom_image = decode_latents(latents)
elapsed = time.perf_counter() - start

print(f"Custom sampler runtime: {elapsed:.1f}s")
plt.figure(figsize=(6, 6))
plt.imshow(custom_image)
plt.axis("off")
plt.title("Custom Stable Diffusion sampler")
plt.show()

## 7. Experiment A — Implement a guidance sweep
The **guidance scale** controls how strongly Stable Diffusion follows the text prompt when generating an image.

- **Low scale (1.0):** More freedom for the model, but the image may not closely match the prompt.
- **Medium scale (3.0):** A balance between creative freedom and prompt adherence.
- **Higher scale (7.5):** Stronger adherence to the prompt, often producing a good balance between accuracy and image quality.
- **Very high scale (11.0):** Strong prompt adherence, but potentially less natural images, with exaggerated features or artifacts.

**Key takeaway:** A higher guidance scale does not always mean a better image; it means stronger influence from the text prompt.

Write a helper that calls your generation function for multiple guidance scales and displays the results in one row. Use the **same seed and prompt** for each scale.


In [ ]:
def compare_guidance(prompt, scales, seed=10, steps=20):
    images = []
    for scale in scales:
        latents = generate_latents(
            prompt, num_inference_steps=steps,
            guidance_scale=scale, seed=seed,
        )
        images.append(decode_latents(latents))
    fig, axes = plt.subplots(1, len(scales), figsize=(4 * len(scales), 4))
    if len(scales) == 1:
        axes = [axes]
    for ax, scale, image in zip(axes, scales, images):
        ax.imshow(image); ax.set_title(f"Guidance = {scale}"); ax.axis("off")
    plt.tight_layout(); plt.show()
    return images

compare_guidance(
    "A red bicycle leaning against a cafe in a quiet European street, photorealistic",
    # 🎯TODO: Provide a list of guidance scales to compare.
    [...], seed=10, steps=20,
)


## 8. Experiment B — Compare inference-step counts and runtime

Implement an experiment that varies the number of denoising steps while holding the prompt, seed, and guidance scale fixed. Record runtime. More steps are not guaranteed to improve the result, and scheduler behavior matters.



In [ ]:
def benchmark_steps(prompt, step_counts=(10, 20, 35), seed=10, guidance_scale=7.5):
    results = []
    for step_count in step_counts:
        if device == "cuda":
            torch.cuda.synchronize()
        start = time.perf_counter()
        latents = generate_latents(
            prompt, num_inference_steps=step_count,
            guidance_scale=guidance_scale, seed=seed,
        )
        image = decode_latents(latents)
        if device == "cuda":
            torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
        results.append((step_count, elapsed, image))
        print(f"{step_count} steps: {elapsed:.2f} seconds")

    fig, axes = plt.subplots(1, len(results), figsize=(4 * len(results), 4))
    if len(results) == 1:
        axes = [axes]
    for ax, (steps, seconds, image) in zip(axes, results):
        ax.imshow(image); ax.set_title(f"{steps} steps\n{seconds:.1f}s"); ax.axis("off")
    plt.tight_layout(); plt.show()
    return results

# 🎯TODO: Compare image quality and generation time using different numbers
# of denoising steps.
# Hint: Try a small number of steps first, then increase it to see how
# the image and runtime change.
results = benchmark_steps(
    "A small cabin in a snowy pine forest at sunrise, cinematic landscape",
    step_counts=(...), seed=10, guidance_scale=7.5,
)


## 9. Validate against the Diffusers pipeline

A useful engineering practice is to compare a custom implementation against a trusted reference. The reference pipeline uses the same underlying components and scheduler. With matching settings, the results should be broadly comparable, although exact numerical equality is not guaranteed across versions, precision modes, or implementation details.



In [ ]:
# Compare the custom sampler with the reference Diffusers pipeline.
reference_generator = torch.Generator(device=device).manual_seed(123)
reference_prompt = "A tiny bookstore in a rainy European street, warm lights, cinematic photography"
# 🎯TODO: Generate an image using the reference Diffusers pipeline.
# Hint: Use `pipe()` with the same prompt, image dimensions, number of
# inference steps, and guidance scale as the custom sampler. Use the generator
# defined above
reference = ... .images[0]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(custom_image); axes[0].set_title("Custom sampler"); axes[0].axis("off")
axes[1].imshow(reference); axes[1].set_title("Diffusers pipeline"); axes[1].axis("off")
plt.tight_layout(); plt.show()


## ✨ Bonus challenge

The helper below repeats the same sampling algorithm but records a copy of the latent every `save_every` steps. The final latent is always recorded. This avoids changing the return type of `generate_latents` used by the earlier tasks.
This following cell generates an image while saving intermediate snapshots to visualize how the image gradually emerges during the denoising process.



In [ ]:
@torch.no_grad()
def generate_latents_with_intermediates(
    prompt, height=512, width=512, num_inference_steps=25,
    guidance_scale=7.5, seed=0, save_every=5,
):
    assert height % 8 == 0 and width % 8 == 0
    embeddings = encode_prompt(prompt, tokenizer, text_encoder, device)
    scheduler.set_timesteps(num_inference_steps, device=device)
    generator = torch.Generator(device=device).manual_seed(seed)
    latents = torch.randn(
        (1, unet.config.in_channels, height // 8, width // 8),
        generator=generator, device=device, dtype=embeddings.dtype,
    ) * scheduler.init_noise_sigma
    snapshots = []
    for i, t in enumerate(scheduler.timesteps):
        model_input = scheduler.scale_model_input(torch.cat([latents, latents]), t)
        noise = unet(model_input, t, encoder_hidden_states=embeddings).sample
        uncond, cond = noise.chunk(2)
        guided = uncond + guidance_scale * (cond - uncond)
        latents = scheduler.step(guided, t, latents).prev_sample
        if (i + 1) % save_every == 0 or i == len(scheduler.timesteps) - 1:
            snapshots.append((i + 1, latents.detach().clone()))
    return latents, snapshots

intermediate_prompt = "A red fox sitting in a mossy forest, soft morning light, detailed illustration"
# 🎯TODO: Run the sampler and save intermediate latent snapshots.
# Hint: Use generate_latents_with_intermediates() with 25 inference steps, seed 42, and save every 5 steps.
final_latents, snapshots = ...
fig, axes = plt.subplots(1, len(snapshots), figsize=(4 * len(snapshots), 4))
if len(snapshots) == 1:
    axes = [axes]
for ax, (step_index, latent_snapshot) in zip(axes, snapshots):
    ax.imshow(decode_latents(latent_snapshot))
    ax.set_title(f"After step {step_index}")
    ax.axis("off")
plt.tight_layout(); plt.show()


## 💡 The Big Picture: Stable Diffusion Pipeline

Stable Diffusion combines text encoding, noise prediction, iterative sampling, and image decoding to transform a text prompt into an image.

```text
       📝 Text Prompt
              |
              ▼
       🔤 Text Encoder
    Converts prompt into
      text embeddings
              |
              ▼
       🎲 Initial Noise
    Create random noise in
       latent space
              |
              ▼
    ┌─────────────────────┐
    │  🔄 Iterative        │
    │     Denoising        │
    │                     │
    │  1. U-Net predicts  │
    │     noise           │
    │          |          │
    │          ▼          │
    │  2. Apply Classifier│
    │     Free Guidance   │
    │     (CFG)            │
    │          |          │
    │          ▼          │
    │  3. Scheduler        │
    │     updates latent   │
    │          |          │
    │          └──────┐    │
    │                 │    │
    │     Repeat for each  │
    │     denoising step   │
    └──────────┬──────────┘
               |
               ▼
        🖼️ VAE Decoder
     Converts final latent
       into pixel space
               |
               ▼
       ✨ Generated Image
```

### 🔍 What Happens at Each Stage?

1. **📝 Text prompt:** Describes the image we want to generate.
2. **🔤 Text encoder:** Converts the prompt into numerical embeddings that guide image generation.
3. **🎲 Initial noise:** Starts the process with random noise in a compact latent representation.
4. **🔄 Iterative denoising:** The U-Net predicts the noise at each timestep. Classifier-free guidance combines unconditional and text-conditioned predictions, while the scheduler uses the result to update the latent. This process repeats for the selected number of inference steps.
5. **🖼️ VAE decoder:** Converts the final denoised latent into an image in pixel space.
6. **✨ Generated image:** The resulting image can be displayed or saved.

### 🎛️ What Influences the Result?

- **Prompt:** Determines the content and characteristics we want in the image.
- **Guidance scale:** Controls how strongly the generation follows the prompt. A higher value is not always better.
- **Inference steps:** More steps can improve image quality, but increase computation time, and improvements may eventually plateau.
- **Scheduler:** Determines how the latent is updated at each denoising step.
- **Random seed:** Helps reproduce results when the other settings remain the same.
- **Intermediate snapshots:** Show how the latent representation gradually changes during denoising.

### 🎯 Key Takeaway

Text embeddings guide the generation process, the U-Net predicts noise, the scheduler progressively updates the latent representation, and the VAE decoder transforms the final latent into an image. Understanding these components helps us reason about image quality, computational cost, and the effects of different sampling choices.